In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
TOKENIZER_PATH = '../tokenizer.pickle'
XGB_PATH       = '../xgb.pkl'
TOP_N          = 50


In [ ]:
with open(TOKENIZER_PATH, 'rb') as f:
    tokenizer = pickle.load(f)

with open(XGB_PATH, 'rb') as f:
    model = pickle.load(f)

# word_index is token -> int, invert to int -> token
index_to_word = {v: k for k, v in tokenizer.word_index.items()}
print(f'Vocabulary size: {len(index_to_word)}')

In [ ]:
scores = model.feature_importances_

# texts_to_matrix columns correspond to token indices 0..MAX_NB_WORDS
# index 0 is unused padding, so token i maps to column i
rows = []
for col_idx, score in enumerate(scores):
    if score > 0 and col_idx in index_to_word:
        rows.append({'token': index_to_word[col_idx], 'importance': score})

df = pd.DataFrame(rows).sort_values('importance', ascending=False).reset_index(drop=True)
print(f'Non-zero features: {len(df)}')
df.head(TOP_N)

In [ ]:
top = df.head(TOP_N)
plt.figure(figsize=(10, TOP_N * 0.3))
plt.barh(top['token'][::-1], top['importance'][::-1])
plt.xlabel('Importance')
plt.title(f'Top {TOP_N} tokens by XGB feature importance')
plt.tight_layout()
plt.show()